# Figure: Smoothed ReLU under Multiplicative Gaussian Input Noise

Validates the closed-form expectation of the ReLU under Gaussian masking:
$$\mathbb{E}_{\mathbf{c}\sim\mathcal{N}(\mathbf{1},\kappa^2\mathbf{I})}\!\big[\sigma(\mathbf{w}^\top(\mathbf{x}\odot\mathbf{c}))\big] = z\,\Phi(z/\sigma) + \sigma\,\varphi(z/\sigma),\quad z=\mathbf{w}^\top\mathbf{x},\ \sigma=\kappa\|\mathbf{w}\odot\mathbf{x}\|_2$$

and compares the closed form against a Monte-Carlo estimate. **Pure NumPy/SciPy — runs in seconds, no GPU needed.**

Outputs `sim1_vary_wtx.png/pdf` (proxy $\hat{\sigma}$) and `sim1_vary_wtx_optionA_exact.png/pdf` (exact $\tilde{\sigma}$).


In [ ]:
# plots the proxy  σ̂(z)= z * Φ(z/(κ||w⊙x||))  and matches it empirically

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm as scipy_norm

# Colab-safe serif + math
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif', 'Times New Roman', 'Times']
plt.rcParams['mathtext.fontset'] = 'cm'

np.random.seed(42)

def relu(z):
    return np.maximum(0.0, z)

# ----- Settings to match Figure 2 -----
kappa_fixed = 0.2
fixed_norm_u = 1.0                 # corresponds to ||w ⊙ x||_2 in Fig. 3 caption
sigma = kappa_fixed * fixed_norm_u

# Dense grid so the curve is visibly smooth
z_grid = np.linspace(-2.1, 2.1, 400)

# ----- Theoretical (proxy) σ̂: passes through (0,0) -----
theory = z_grid * scipy_norm.cdf(z_grid / sigma)

# ----- Empirical Monte Carlo of the SAME quantity: E[ (Z) 1{Z>=0} ] -----
# where Z = z + epsilon, epsilon ~ N(0, sigma^2)
n_samples = 200000
eps = sigma * np.random.randn(n_samples)                 # (N,)
Z = eps[:, None] + z_grid[None, :]                       # (N, |z_grid|)
empirical = (Z * (Z >= 0)).mean(axis=0)

# Standard ReLU baseline
relu_vals = relu(z_grid)

plt.figure(figsize=(16, 8))
plt.plot(
    z_grid, theory,
    'o-', markevery=25, markersize=10, linewidth=4,
    label=f'Theoretical $\\hat{{\\sigma}}(\\mathbf{{w}}, \\mathbf{{x}})$ ($\\kappa={kappa_fixed}$)'
)
plt.plot(
    z_grid, empirical,
    's--', markevery=25, markersize=10, linewidth=4,
    label=f'Empirical $\\mathbb{{E}}_{{\\mathbf{{c}}}}[\\sigma(\\mathbf{{w}}^T(\\mathbf{{x}} \\odot \\mathbf{{c}}))]$ ($\\kappa={kappa_fixed}$)'
)
plt.plot(
    z_grid, relu_vals,
    'k:', linewidth=4,
    label='Standard ReLU $\\sigma(\\mathbf{w}^T\\mathbf{x})$'
)

plt.xlabel('Input $\\mathbf{w}^T\\mathbf{x}$', fontsize=30)
plt.ylabel('Activation Value', fontsize=30)
plt.xticks(fontsize=30)
plt.yticks(fontsize=30)
plt.legend(fontsize=30)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("sim1_vary_wtx.png", dpi=300)
plt.savefig("sim1_vary_wtx.pdf")
plt.show()


In [ ]:
# Option A: Theoretical curve = EXACT expectation
#   E_c[ ReLU(w^T (x ⊙ c)) ]  with  c ~ N(1, κ^2 I)
# which equals (for z = w^T x and σ = κ ||w ⊙ x||_2):
#   z * Φ(z/σ) + σ * φ(z/σ)
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm as scipy_norm

# Colab-safe serif
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['DejaVu Serif', 'Times New Roman', 'Times']
plt.rcParams['mathtext.fontset'] = 'cm'

np.random.seed(42)

def relu(z):
    return np.maximum(0.0, z)

kappa_fixed = 0.2

# fix ||w ⊙ x||_2 to isolate κ effect
fixed_norm_u = 1.0
sigma = kappa_fixed * fixed_norm_u

# Dense grid so the curve is visibly smooth
z_grid = np.linspace(-2.1, 2.1, 400)

# ---- OPTION A THEORY: exact expectation of masked ReLU ----
# E[ReLU(N(z, sigma^2))] = z Φ(z/sigma) + sigma φ(z/sigma)
theory = z_grid * scipy_norm.cdf(z_grid / sigma) + sigma * scipy_norm.pdf(z_grid / sigma)

# ---- Empirical Monte Carlo for the SAME quantity ----
n_samples = 200000
eps = sigma * np.random.randn(n_samples)                  # (N,)
Z = eps[:, None] + z_grid[None, :]                        # (N, |z_grid|)
empirical = relu(Z).mean(axis=0)

# Standard ReLU baseline
relu_vals = relu(z_grid)

plt.figure(figsize=(16, 8))

plt.plot(
    z_grid, theory,
    'o-', markevery=25, markersize=10, linewidth=4,
    label=f'Theoretical $\\tilde{{\\sigma}}(\\mathbf{{w}}, \\mathbf{{x}})$ ($\\kappa={kappa_fixed}$)'
)
plt.plot(
    z_grid, empirical,
    's--', markevery=25, markersize=10, linewidth=4,
    label=f'Empirical $\\mathbb{{E}}_{{\\mathbf{{c}}}}[\\sigma(\\mathbf{{w}}^T(\\mathbf{{x}} \\odot \\mathbf{{c}}))]$ ($\\kappa={kappa_fixed}$)'
)
plt.plot(
    z_grid, relu_vals,
    'k:', linewidth=4,
    label='Standard ReLU $\\sigma(\\mathbf{w}^T\\mathbf{x})$'
)

plt.xlabel('Input $\\mathbf{w}^T\\mathbf{x}$', fontsize=30)
plt.ylabel('Activation Value', fontsize=30)
plt.xticks(fontsize=30)
plt.yticks(fontsize=30)
plt.legend(fontsize=30)
plt.grid(True, linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig("sim1_vary_wtx_optionA_exact.png", dpi=300)
plt.savefig("sim1_vary_wtx_optionA_exact.pdf")
plt.show()
